# Introduction
Urban Rivers is a conservation organization helping to restore the Chicago River.  

Part of the project involves tracking changes in biodiversity attributable to the installation of floating wetlands.  
Volunteers have placed and maintain motion detection cameras (camera traps) along installations, natural river banks, and the existing metal retaining walls on the water way.

These pictures are available on s3 - this code investigates downloading a portion of those images for testing with SpeciesNet.

> Removed geofencing, using hashmd5 for image names, full dataset, clean when finished.  
> Connects to production mongo.  
> Updates and persists json and csv files for continual processing into master json.  
> 2025-07-06 Testing with min 1280px width in 10k batches


This workbook was used for detections using Kaggle's GPUs and is linked therein.  
https://www.kaggle.com/code/morescope/speciesnet-testing-urbanrivers

## Notebook Setup and Required Packages

In [1]:
# Data Handling
import pandas as pd
import numpy as np

# IO - getting files and images from MongoDB and S3
from pymongo import MongoClient
from kaggle_secrets import UserSecretsClient
import requests

from concurrent.futures import ThreadPoolExecutor, as_completed

from pathlib import Path
from PIL import Image
from io import BytesIO

import os
import sys
import re
import shutil
import json
import time
from datetime import datetime

# Move speciesnet install to where it's used

from IPython.display import display, HTML, JSON, Javascript

import kagglehub

print("Libraries Loaded")

Libraries Loaded


In [2]:
# Install speciesnet and related megadetector libraries
!pip install --quiet speciesnet megadetector-utils

print("Loaded speciesnet and megadetector")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 29.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.5/97.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 687.4/687.4 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 422.7/422.7 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 956.3/956.3 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.0/81.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.6/94.6 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.7/111.7 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0

In [3]:
# Configuration for Multithreading and Batching
num_batches = 10
max_threads = 20

# Prepare folders
output_root = Path("output")
output_root.mkdir(exist_ok=True)
images_root = Path("images")
images_root.mkdir(exist_ok=True)

In [4]:
# Add persistent data if available
!cp /kaggle/input/ur-speciesnet-persistent/output/predictions_dict_master.json /kaggle/working/output/predictions_dict_master.json

## Access The URIs from S3 through MongoDB

In [5]:
# Get the stored mongo uri secret
user_secrets = UserSecretsClient()
mongo_uri = user_secrets.get_secret("MONGO_PROD")

# Connect to the MongoDB client
client = MongoClient(mongo_uri)
 
# Access the database and collection
db = client['test']
collection = db['cameratrapmedias'] 
 
# Query the collection to retrieve records with image URLs, metadata, and the first index of 'relativePath'
data = list(collection.aggregate([
    {
        '$match': {
            'aiResults': None  # Match documents where aiResults is null or missing
        }
    },
    {
        '$project': {
            '_id': 0,
            'publicURL': 1,
            'timestamp': 1,
            'folderName': { '$arrayElemAt': ['$relativePath', 1] },
            'fileName': 1,
            'mediaID': 1
        }
    },
    # { '$limit': 150 }
]))

# Stop execution if no records are found
if not data:
    display(HTML("<h3 style='color:red;'>No records found. Exiting notebook.</h3>"))
    sys.exit()
    !exit # Redundant full exit
 
# Convert the data to a pandas DataFrame for exploration
df = pd.DataFrame(data)

# preview df
display(df.sort_values(by='timestamp', ascending=False).head())
print(f'Rows: {len(df)}')

,mediaID,timestamp,publicURL,fileName,folderName
195555,b3ceeb9972bf870712299cd51284538e,2026-02-21 12:35:28,https://urbanriverrangers.s3.amazonaws.com/ima...,PICT0373.JPG,2026-02-21_UR004
195182,5c137aac6c88e672505b46f2590b6190,2026-02-21 12:35:27,https://urbanriverrangers.s3.amazonaws.com/ima...,HDPH0373.JPG,2026-02-21_UR004
195554,1f595dbfb3122ded2d79866086bf2a76,2026-02-21 11:16:01,https://urbanriverrangers.s3.amazonaws.com/ima...,PICT0372.JPG,2026-02-21_UR004
195181,d99fcce6a80777d7161cb63bc2878350,2026-02-21 11:16:00,https://urbanriverrangers.s3.amazonaws.com/ima...,HDPH0372.JPG,2026-02-21_UR004
195553,8c4826e85bdb830cbf4be4775d55bc8c,2026-02-21 11:14:55,https://urbanriverrangers.s3.amazonaws.com/ima...,PICT0371.JPG,2026-02-21_UR004


Rows: 196479


In [6]:
# Export the production array to a CSV file 
df.to_csv('ur_test_medias_prod.csv', index=False)

In [7]:
# load the existing predictions_dict to filter out media IDs already predicted
predictions_file = Path("output/predictions_dict_master.json")
with open(predictions_file, "r") as f:
    predictions_data = json.load(f)

# Extract mediaIDs from filepaths in predictions
predicted_filepaths = [p["filepath"] for p in predictions_data.get("predictions", [])]
predicted_media_ids = {Path(fp).stem for fp in predicted_filepaths}

print(f"Found {len(predicted_media_ids)} predicted mediaIDs")

# Filter out rows where mediaID is already in the predictions
initial_count = len(df)
df_filtered = df[~df["mediaID"].astype(str).isin(predicted_media_ids)]
filtered_count = len(df_filtered)

print(f"Filtered out {initial_count - filtered_count} rows. Remaining: {filtered_count}")

Found 532190 predicted mediaIDs
Filtered out 100000 rows. Remaining: 96479


In [8]:
# Save the filtered df
df_filtered.to_csv("ur_test_medias_prod_filtered.csv", index=False)

# Filtered out files to download and predict
The persistent files for predictions.json that continue to grow will serve as a means of not redownloading images.
Now that we have a connection to the MongoDB server and access to the URLs, let's use the download images.

# Max Images at 1280px resolution
Trial and error puts this at some point after 70k - so we'll back off to 50k to be conservative

In [9]:
# Each time running - just process the next 10k files
if filtered_count > 50000:
    df_to_run = df_filtered[:50000]
else:
    df_to_run = df_filtered

## Download Images

In [10]:
%%time
# Create a directory to save the images - redundant but ok if in testing
output_root.mkdir(exist_ok=True)
path = Path('images')
path.mkdir(exist_ok=True)

# Optional - define chunks - for each run, the first n rows will be processed
df_download = df_to_run # up to 50k images based on total amount remaining
print(f'Peparing to Download {len(df_download)} images')

# Create a tool for resizing so cropping top and bottom can happen while keeping the aspect ratio
def resize_to_height(image, target_width=1280):
    og_width, og_height = image.size
    new_height = int(og_height * (target_width / og_width))
    return image.resize((target_width, new_height))

# Tool for download and processing
def process_row(row, dest_folder, session):
    url = row['publicURL']
    filename = f"{row['mediaID']}.jpg"
    dest = dest_folder / filename

    try:
        response = session.get(url, timeout=5)
        response.raise_for_status()

        image = Image.open(BytesIO(response.content)).convert("RGB")
        image = resize_to_height(image, target_width=1280)
        image.save(dest, format="JPEG", quality=75)
    except Exception as e:
        print(f"failed to process {filename}: {e}")

for batch_idx, df_chunk in enumerate(np.array_split(df_download, num_batches)):
    batch_folder = images_root / f'batch_{batch_idx}'
    batch_folder.mkdir(exist_ok=True)
    print(f'Processing batch {batch_idx + 1} / {num_batches}...')

    rows = df_chunk.to_dict(orient='records')
    start = time.time()

    with requests.Session() as session:
        with ThreadPoolExecutor(max_workers=max_threads) as executor:
            futures = [executor.submit(process_row, row, batch_folder, session) for row in rows]
            for future in as_completed(futures):
                future.result()  # you can add error catching here if needed

    print(f"Batch {batch_idx+1} took {time.time() - start:.2f} seconds.")
        
print(f'{len(df_download)} Images Downloaded and Resized')

Peparing to Download 50000 images
Processing batch 1 / 10...


/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


failed to process 0576369af035c23d60d4b0bb454b6fcc.jpg: image file is truncated (12 bytes not processed)
failed to process d41d8cd98f00b204e9800998ecf8427e.jpg: cannot identify image file <_io.BytesIO object at 0x7a02f17312b0>failed to process fa124d2eb205e4adb18d8e56f1670cf9.jpg: cannot identify image file <_io.BytesIO object at 0x7a02f174d9e0>
failed to process 3fe52eabcaf2a5ea3ddfe5d70277feaf.jpg: cannot identify image file <_io.BytesIO object at 0x7a02e852b240>

failed to process 0bea2ca4e25ec9d5984caaea393897f4.jpg: cannot identify image file <_io.BytesIO object at 0x7a02e8562610>
failed to process 8675c6e4b47fb1f72aa9ccf4a5a141f6.jpg: cannot identify image file <_io.BytesIO object at 0x7a02e8560bd0>
failed to process e764485e7bb9d3c81e2e231accb96644.jpg: cannot identify image file <_io.BytesIO object at 0x7a02e84b8cc0>
failed to process 70e3354e57a23095aa89e20385d6b3e8.jpg: cannot identify image file <_io.BytesIO object at 0x7a02e8560a90>
failed to process 39e988c52f38a6072edfd3c

In [11]:
# Uncomment and run this if the images need to be redone
# !rm images -r
# !rm output/docs -r
# !rm docs.zip
# %lsmagic

## Running Species Net on the Full Dataset
Now that we have the max number of images downloaded (19.5GB) let's run speciesnet

Note there might be a better way of doing this using bytes downloaded from s3 - but I haven't figured that part out yet.

### We're going to try a multithreading chunks approach

In [12]:
def print_predictions(predictions_dict: dict) -> None:
    print("Predictions:")
    for prediction in predictions_dict["predictions"][0:1]:
        print(prediction["filepath"], "=>", prediction["prediction"])

### Download Model

In [13]:
# Choose the folder we're going to download the model to
model_path = '/content/models'
os.makedirs(model_path, exist_ok=True)

# Download the model (it will go to a folder like /kaggle/input/...)
download_path = kagglehub.model_download('google/speciesnet/PyTorch/v4.0.1a',
                                          force_download=True)

print('Model downloaded to temporary folder: {}'.format(download_path))

# List the contents of the downloaded directory to identify the actual files/subdirectories
model_files = os.listdir(download_path)

# Copy the contents of the model file to our destination folder
for item_name in model_files:
    source_path = os.path.join(download_path, item_name)
    destination_path = os.path.join(model_path, item_name)
    if os.path.isfile(source_path):
        shutil.copy2(source_path, destination_path)
    elif os.path.isdir(source_path):
        shutil.copytree(source_path, destination_path, dirs_exist_ok=True)

print('{} files copied to: {}'.format(len(model_files),model_path))

Model downloaded to temporary folder: /kaggle/input/speciesnet/pytorch/v4.0.1a/1
6 files copied to: /content/models


In [14]:
# Pick the model we want to use (4.0.1a)
!pip install protobuf==3.20.3
from speciesnet import SpeciesNet 
model = SpeciesNet(model_path)

print('Model Loaded')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 4.3 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
onnx 1.20.1 requires protobuf>=4.25.1, but you have protobuf 3.20.3 which is incompatible.
a2a-sdk 0.3.23 requires protobuf>=5.29.5, but you have protobuf 3.20.3 which is incompatible.
grain 0.2.15 requires protobuf>=5.28.3, but you have protobuf 3.20.3 which is incompatible.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.
tensorflow-m

In [15]:
# Let's format a request string as a list of dicts (aka JSON string format)
def create_instances(batch_folder):
    image_paths = [f'{batch_folder}/{f}' for f in os.listdir(batch_folder) if f.lower().endswith('.jpg')]

    instances = []
    for image_path in image_paths:
        instances.append({
            'filepath': image_path
        })

    # Check that it's saved correctly by verifying the first
    print(instances[0:1])

    return instances


for batch_index in range(len(os.listdir(images_root))):
    instances = create_instances(f'{images_root}/batch_{batch_index}')

    # make the predictions and get a sense of how long it would take
    %time predictions_dict = model.predict(instances_dict={"instances": instances})

    print_predictions(predictions_dict) # show the first prediction of each batch

    # Save the dict to the batch folder
    with open(f'{images_root}/batch_{batch_index}/predictions_dict_{batch_index}.json', 'w') as f:
        json.dump(predictions_dict, f, indent=2)

    print(f'predictions_dict_{batch_index}.json saved to {images_root}/batch_{batch_index}')

[{'filepath': 'images/batch_0/586bebff45e98ec4c6a8eef0f4eba372.jpg'}]
CPU times: user 20min 14s, sys: 21.5 s, total: 20min 35s
Wall time: 9min 13s
Predictions:
images/batch_0/586bebff45e98ec4c6a8eef0f4eba372.jpg => b1352069-a39c-4a84-a949-60044271c0c1;aves;;;;;bird
predictions_dict_0.json saved to images/batch_0
[{'filepath': 'images/batch_1/c16cb1ba82b6c29233426eedad03c145.jpg'}]
CPU times: user 20min 56s, sys: 14.5 s, total: 21min 11s
Wall time: 9min 32s
Predictions:
images/batch_1/c16cb1ba82b6c29233426eedad03c145.jpg => f1856211-cfb7-4a5b-9158-c0f72fd09ee6;;;;;;blank
predictions_dict_1.json saved to images/batch_1
[{'filepath': 'images/batch_2/853bc4410a51f591ad2367bc646e7958.jpg'}]
CPU times: user 20min 52s, sys: 20.7 s, total: 21min 12s
Wall time: 9min 38s
Predictions:
images/batch_2/853bc4410a51f591ad2367bc646e7958.jpg => f1856211-cfb7-4a5b-9158-c0f72fd09ee6;;;;;;blank
predictions_dict_2.json saved to images/batch_2
[{'filepath': 'images/batch_3/8e08aa449f6f43c8e0cb9bc0dca18687.j

## Let's save the predictions dict json file

In [16]:
%%time
# To concatenate all the json files
output_file = output_root / "predictions_dict_master.json" # Uncomment when loading from start

# Initialize the master predictions list
master_predictions = []

# Load existing master file if it exists
if output_file.exists():
    with open(output_file, "r") as f:
        existing_data = json.load(f)
        if "predictions" in existing_data:
            master_predictions.extend(existing_data["predictions"])
        else:
            print(f"{output_file} missing 'predictions' key")

# Use today's date
rundate = datetime.now().date().isoformat()

# initialize the current run list
current_predictions = []

# Loop through files matching the pattern
for json_file in sorted(images_root.glob("batch_*/predictions_dict_*.json")):
    with open(json_file, "r") as f:
        data = json.load(f)
        if "predictions" in data:
            for record in data["predictions"]:
                record["run_date"] = rundate # adding a run_date field to each predictions record
                master_predictions.append(record)  # Concatenate predictions!
                current_predictions.append(record) # Concat current preds too
        else:
            print(f"{json_file} missing 'predictions' key")

# Deduplicate based on media ID extracted from 'filepath'
deduped_predictions = {
    Path(record["filepath"]).stem: record
    for record in master_predictions
}
master_predictions = list(deduped_predictions.values())

# Deduplicate current preds based on media ID extracted from 'filepath'
current_deduped_predictions = {
    Path(record["filepath"]).stem: record
    for record in current_predictions
}
current_predictions = list(current_deduped_predictions.values())

# Write the combined predictions to a new file
with open(output_file, "w") as f:
    json.dump({"predictions": master_predictions}, f, indent=2)

print(f"Combined {len(master_predictions)} predictions into {output_file}")

# Write the current preds to a new file
with open(f'{output_root}/current_predictions_dict.json', 'w') as f:
    json.dump({"predictions": current_predictions}, f, indent=2)

print(f"Saved {len(current_predictions)} current predictions in current_predictions_dict.json ")

Combined 581976 predictions into output/predictions_dict_master.json
Saved 49786 current predictions in current_predictions_dict.json 
CPU times: user 51.5 s, sys: 4.15 s, total: 55.7 s
Wall time: 58.4 s


### Final Cleanup of Files
Remove all images because we are adding files persistence

In [17]:
# Remove the image directories because nobody needs to store them at the end here
shutil.rmtree('/kaggle/working/images')

print("Files cleaned up")

Files cleaned up
